# Analiza upitnika — jednostavna verzija (pravi podaci)

**Kolegij: Data Science u kulturi · Filozofski fakultet u Rijeci**
Nositelj: izv. prof. dr. sc. Benedikt Perak

Ovo je kratka verzija analize upitnika „Gdje se izlazi van u Rijeci". Radi na **pravim odgovorima** iz objavljene Google tablice i sastoji se od osam malih koraka. Svaki korak ima kratko objašnjenje i nekoliko redaka koda — ništa više.

Osam koraka: (1) poveznica na podatke, (2) pogled u tablicu, (3) kraća imena stupaca, (4) čišćenje dviju stvari, (5) brojanje odgovora, (6) slobodni tekst, (7) skale od 1 do 5, (8) kratki sažetak.

Na kraju je uputa **kako iste korake primijeniti na vlastiti upitnik** — za to je ova bilježnica i napravljena.

Pokreni ćelije redom (Shift+Enter). Sve radi bez ikakvih ključeva i bez instaliranja.


## Korak 1 — Poveznica na podatke

Prvo pitanje u svakoj analizi: **odakle podaci ulaze?** Kod Google obrasca to je objavljena tablica s odgovorima.

Kako se to napravi (jednom, i vrijedi zauvijek): u obrascu otvori *Responses (Odgovori)* → *Link to Sheets* → nastane tablica. U tablici *File → Share → Publish to web*, odabereš list s odgovorima i format **CSV**, klikneš *Publish*. Dobiješ poveznicu koja završava s `pub?output=csv`.

Ta poveznica je „živa": svaki novi odgovor u obrascu odmah se vidi i ovdje. Kod se zato nikad ne mijenja — samo ponovno pokreneš ćeliju.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Poveznica na objavljenu tablicu (obrazac 'Gdje se izlazi van u Rijeci').
# Za svoj upitnik: zamijeni ovaj tekst svojom poveznicom koja zavrsava na 'pub?output=csv'.
POVEZNICA = ('https://docs.google.com/spreadsheets/d/e/'
             '2PACX-1vRI07LBNnv5uMnf65Iqymsed70JEdyhB_wVNMjMVfzbzSt9Yf_8F317WD5rAYnaUhwUko7qUYgFNeAE'
             '/pub?output=csv')

podaci = pd.read_csv(POVEZNICA, dtype=str)
print('Ucitano redaka:', len(podaci))


Ako je tablica objavljena u formatu **xlsx** (poveznica završava s `pub?output=xlsx`), ista dva retka izgledaju ovako:

```python
podaci = pd.read_excel(POVEZNICA)   # umjesto read_csv
```

`pandas` prepoznaje format po sadržaju, pa je razlika samo u imenu funkcije. CSV je jednostavniji i brži, pa ćemo njega koristiti dalje.


## Korak 2 — Pogled u tablicu

Prije bilo kakvog računanja treba vidjeti **što je u tablici**. Gledamo tri stvari: koliko ima redaka (to je broj ispitanika), kako se zovu stupci i kako izgledaju prvi odgovori.

Zapamti: imena stupaca Google prepisuje **doslovno iz teksta pitanja**, pa su zato tako dugačka i ponekad ružna. To nije greška — to je doslovni prijepis.


In [ ]:
print('Broj ispitanika:', len(podaci))
print('Broj stupaca:', podaci.shape[1])
print()
for i, ime in enumerate(podaci.columns, 1):
    print(i, '.', ime)
print()
podaci.head(5)


## Korak 3 — Kraća imena stupaca

Dugačka imena otežavaju rad (svaki poziv na stupac postaje dugačak redak i lako se pogriješi slovo). Zato svakom stupcu damo **kratko ime** — i to tako da ga tražimo po ključnoj riječi, a ne po točnom tekstu.

Zašto po ključnoj riječi: ako predavač kasnije popravi tipfeler u pitanju, ime stupca se promijeni, a naš kod ostane ispravan.


In [ ]:
def nadji(knjiga, *kljucevi):
    'Vraca ime stupca koje sadrzi neku od kljucnih rijeci.'
    for k in kljucevi:
        for ime in knjiga.columns:
            if k.lower() in str(ime).lower():
                return ime
    return None


kratka_imena = {}
parovi = [
    ('mjesta', ('izlazite', 'izlazi')),
    ('dob', ('dob',)),
    ('glazba', ('glazbu', 'glazba')),
    ('narodnjaci', ('narodnjak',)),
    ('bach', ('bach',)),
]
for kratko, kljucevi in parovi:
    stupac = nadji(podaci, *kljucevi)
    if stupac:
        kratka_imena[stupac] = kratko

podaci = podaci.rename(columns=kratka_imena)
print('Prepoznati stupci:', kratka_imena)
print('Svi stupci sada:', list(podaci.columns))


## Korak 4 — Čišćenje: dvije stvari koje uvijek trebaju popravak

**Prvo: broj zapisan kao tekst.** Google šalje sve kao tekst, čak i dob. Pretvaramo je u broj, a ono što se ne može pretvoriti (npr. „dvadeset") ostaje prazno i **isključuje se iz izračuna sredine** — nikako ne postaje nula, jer bi nula pokvarila prosjek.

**Drugo: više odgovora u jednoj ćeliji.** Kod pitanja s kvačicama u jednoj ćeliji stoji npr. `Kino Cinestar, Klub Crkva, Kafe Dnevni boravak`. Takav zapis ne smijemo brojiti kao jedan odgovor, nego ga rastavimo na popis i tek onda brojimo svaki dio posebno.


In [ ]:
# 1) dob u broj
podaci['dob_broj'] = pd.to_numeric(podaci['dob'], errors='coerce')
print('Dob: prepoznato kao broj:', int(podaci['dob_broj'].notna().sum()), 'od', len(podaci), 'odgovora')
print('Vrijednosti koje nisu broj:', podaci.loc[podaci['dob_broj'].isna(), 'dob'].tolist())

# 2) mjesta kao popis
podaci['mjesta_popis'] = podaci['mjesta'].apply(
    lambda redak: [d.strip() for d in str(redak).split(',') if d.strip()])
print('Prosjek odabranih mjesta po osobi:', round(podaci['mjesta_popis'].apply(len).mean(), 2))
print('Primjer:', podaci['mjesta_popis'].iloc[0])


## Korak 5 — Brojanje odgovora (frekvencije)

Sada brojimo koliko je puta odabrano svako mjesto. Uz **broj** uvijek gledamo i **postotak**, jer broj sam ne govori ništa: 6 odabira je puno ako imamo 18 ispitanika, a malo ako ih imamo 300.

Pazi na jednu stvar: postoci se zbrajaju **preko 100 %** i to je u redu. Svatko je mogao odabrati više mjesta, pa jedna osoba ulazi u više stupaca.


In [ ]:
svi_odabiri = [m for popis in podaci['mjesta_popis'] for m in popis]
brojevi = pd.Series(Counter(svi_odabiri)).sort_values(ascending=False)

pregled = pd.DataFrame({'mjesto': brojevi.index, 'odabira': brojevi.values})
pregled['posto_ispitanika'] = [round(100 * n / len(podaci), 1) for n in brojevi.values]
pregled


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
y = list(range(len(brojevi)))[::-1]
ax.barh(y, brojevi.values, color='#22d3ee', height=0.6)
ax.set_yticks(y)
ax.set_yticklabels(brojevi.index, fontsize=9)
for yi, vrijednost in zip(y, brojevi.values):
    ax.text(vrijednost + 0.1, yi, str(vrijednost), va='center', fontsize=9)
ax.set_xlabel('broj ispitanika (od ' + str(len(podaci)) + ')')
ax.set_title('Gdje se izlazi u Rijeci')
plt.tight_layout()
plt.show()


## Korak 6 — Slobodni tekst

Polje „Koju glazbu slušate" je otvoreno pitanje i tu se vidi zašto je priprema podataka najveći dio posla. Ljudi su pisali i zareze i točke sa zarezom, velika i mala slova, a neki su umjesto izvođača naveli **žanr** („Punk, rock, trap").

Zato: rastavi zapis na dijelove, skini razmake, sve svedi na mala slova — pa broji. Tako se „Rock", „rock" i „ rock" spoje u jedan odgovor.


In [ ]:
import re

def na_dijelove(tekst):
    if pd.isna(tekst):
        return []
    dijelovi = re.split(r';|,|/|\|| i ', str(tekst))
    return [d.strip().lower() for d in dijelovi if len(d.strip()) > 1]

podaci['glazba_popis'] = podaci['glazba'].apply(na_dijelove)
najcesci = pd.Series(Counter([g for popis in podaci['glazba_popis'] for g in popis]))
najcesci = najcesci.sort_values(ascending=False)

print('Različitih zapisa u stupcu:', sum(1 for p in podaci['glazba_popis'] for _ in p))
print('Nakon spajanja istih imena:', len(najcesci))
print()
top = najcesci.head(12)[::-1]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(list(top.index), top.values, color='#a78bfa', height=0.6)
for i, v in enumerate(top.values):
    ax.text(v + 0.05, i, str(v), va='center', fontsize=9)
ax.set_xlabel('broj navoda')
ax.set_title('Najčešće navedeno u pitanju o glazbi')
plt.tight_layout()
plt.show()


## Korak 7 — Skale od 1 do 5

Dvije tvrdnje ocjenjivane su od 1 (uopće se ne slažem) do 5 (u potpunosti se slažem). Ovdje vrijedi važno pravilo: **broj nije mjera nego oznaka stupnja slaganja** — zato uz sredinu uvijek gledamo i cijelu raspodjelu. Dvije tvrdnje mogu imati istu sredinu, a potpuno različite odgovore: jedna znači „svi su neutralni", druga „pola ne, pola da".


In [ ]:
for stupac, naziv in [('narodnjaci', 'narodnjaci u kafićima'), ('bach', 'Bach u slobodno vrijeme')]:
    ocjene = pd.to_numeric(podaci[stupac], errors='coerce').dropna()
    print(naziv, ':')
    print('   ocjene (1-5):', ocjene.astype(int).value_counts().reindex([1, 2, 3, 4, 5]).fillna(0).astype(int).tolist())
    print('   sredina:', round(ocjene.mean(), 2), '| medijan:', int(ocjene.median()), '| ispitanika:', len(ocjene))
    print()


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.2))
stupci = ['narodnjaci', 'bach']
oznake = ['narodnjaci\nu kafićima', 'Bach u slobodno\nvrijeme']
boje = ['#fb7185', '#fbbf24', '#a78bfa', '#34d399', '#22d3ee']
lijevo = [0, 0]
for ocjena in [1, 2, 3, 4, 5]:
    vrijednosti = [int(pd.to_numeric(podaci[s], errors='coerce').dropna().astype(int).eq(ocjena).sum())
                   for s in stupci]
    ax.barh(oznake, vrijednosti, left=lijevo, color=boje[ocjena - 1], label=str(ocjena), height=0.5)
    for i, v in enumerate(vrijednosti):
        if v:
            ax.text(lijevo[i] + v / 2, i, str(v), ha='center', va='center', fontsize=9, color='#0f172a')
    lijevo = [a + b for a, b in zip(lijevo, vrijednosti)]
ax.set_xlabel('broj studenata')
ax.set_title('Slaganje s tvrdnjama (1 = ne slažem se, 5 = slažem se)')
ax.legend(title='ocjena', ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.28))
plt.tight_layout()
plt.show()


## Korak 8 — Kratki sažetak

Zadnji korak: od brojki napraviti tri do četiri rečenice nalaza. To je ono što ide u izvještaj. Pravilo je jednostavno — **svaka rečenica mora se moći provjeriti u tablici iznad**.


In [ ]:
dob = podaci['dob_broj'].dropna()
najbolja_tri = ', '.join(list(brojevi.head(3).index))
najgora = brojevi.index[-1]
narodnjaci = pd.to_numeric(podaci['narodnjaci'], errors='coerce').dropna()
bach = pd.to_numeric(podaci['bach'], errors='coerce').dropna()

print('SAZETAK NALAZA')
print()
print('Upitnik je ispunilo', len(podaci), 'ispitanika, prosječne dobi',
      str(round(dob.mean(), 1)) + ' godina', '(medijan', str(int(dob.median())) + ').')
print('Najčešće se izlazi u:', najbolja_tri + '.')
print('Najrjeđe se bira:', najgora, '(' + str(int(brojevi.values[-1])), 'odabira).')
print('U glazbi se najviše ponavljaju:', ', '.join(list(najcesci.head(4).index)) + '.')
print('Prosječno slaganje s tvrdnjom o narodnjacima je', round(narodnjaci.mean(), 2),
      'a s tvrdnjom o Bachu', round(bach.mean(), 2), 'na skali 1-5.')
print()
print('Ograničenje: uzorak je mali,', len(podaci), 'odgovora. Nalazi vrijede za ovu skupinu,')
print('ne za sve studente u Rijeci.')


## Kako iste korake primijeniti na vlastiti upitnik

Cijela bilježnica radi za bilo koji Google obrazac. Postupak je ovakav:

1. **Napravi obrazac** u Google Formsu. Neka ima barem jedno pitanje s kvačicama, jedno s brojem, jedno slobodnog teksta i dvije tvrdnje na skali 1–5 — tako ćeš proći sve korake iznad.
2. **Poveži odgovore s tablicom**: u obrascu *Responses → Link to Sheets*.
3. **Objavi tablicu**: u tablici *File → Share → Publish to web* → odaberi list s odgovorima → format CSV → *Publish*. Kopiraj poveznicu.
4. **Zalijepi poveznicu** u ćeliju Koraka 1, u varijablu `POVEZNICA`.
5. **U Koraku 3 prilagodi ključne riječi**: umjesto `'izlazite'` upiši riječ koja se pojavljuje u tekstu tvog pitanja (npr. `'najviše'`, `'koristite'`). Kod ispiše koje je stupce prepoznao — provjeri da nijedan nije ostao neprepoznat.
6. Pokreni *Run all* i pogledaj brojke i slike.

Tri stvari koje ćeš gotovo sigurno sresti i na vlastitim podacima:

- **netko će upisati neočekivanu vrijednost** — u ovom upitniku jedan odgovor ima dob 62, pa je prosjek 21,1 godina, a medijan 19; to nije greška koda nego stvarnost podataka, i u izvještaju napiši što si s tim odgovorom napravio;
- **neće svi poštovati uputu o razdvajanju** izvođača točkom sa zarezom — zato kod razdvaja i zareze, kose crte i „i";
- **neće svi odgovoriti na sve** — prazna ćelija nije nula, nego nedostatak odgovora.

Dodatni zadaci: (a) prikaži postotke umjesto brojeva za mjesta; (b) provjeri postoji li razlika u slaganju s tvrdnjama između mlađih i starijih ispitanika; (c) dodaj u upitnik svoje pitanje i ponovi cijeli postupak.

> Ako želiš i korake s umjetnom inteligencijom (kategorizacija žanrova, semantička sličnost ukusa i sažetak koji piše model), oni su u duljoj bilježnici: `colab/DSK_analiza_upitnika_Rijeka.ipynb` u [repozitoriju kolegija](https://github.com/bperak/dsk).

**Materijali:** [repozitorij kolegija](https://github.com/bperak/dsk) · [upitnik](https://forms.gle/wFv6qRhMmqqsKuVo8) · knjiga Perak (2025) *Komunikacija u doba umjetne inteligencije* — [otvoreni pristup](https://github.com/bperak/komunikacija_u_doba_ai)
